# Workshop 3 — Face Matching & Recognition

This notebook demonstrates how to compute embeddings for faces in two images and visualize the best pairings with similarity scores and a thresholded match flag.

**Workflow:**
- Detect faces in `GALLERY_IMAGE` (knowns)
- Detect faces in `QUERY_IMAGE` (to match)
- Compute cosine similarity between embeddings
- Visualize pairs with scores and match status

In [ ]:
# Setup imports and helper functions
import os, sys
proj_root = os.path.abspath(os.path.join('..', "lib"))
if proj_root not in sys.path:
    sys.path.insert(0, proj_root)

import cv2, numpy as np, matplotlib.pyplot as plt
from face_detector import FaceDetector, select_image_file

detector = FaceDetector()

def cosine_sim(a, b):
    a = np.asarray(a, dtype=np.float32)
    b = np.asarray(b, dtype=np.float32)
    if a.ndim == 1:
        a = a.reshape(1, -1)
    if b.ndim == 1:
        b = b.reshape(1, -1)
    dot = np.dot(a, b.T)
    na = np.linalg.norm(a, axis=1)
    nb = np.linalg.norm(b, axis=1)
    return (dot / (na[:,None] * nb[None,:] + 1e-8))

In [ ]:
# Select images
GALLERY_IMAGE = None  # image with known faces (can be a group photo)
QUERY_IMAGE = None    # image with faces to match against gallery

if GALLERY_IMAGE is None:
    print('Select GALLERY_IMAGE (knowns)')
    GALLERY_IMAGE = select_image_file()
if QUERY_IMAGE is None:
    print('Select QUERY_IMAGE (queries)')
    QUERY_IMAGE = select_image_file()

if not GALLERY_IMAGE or not QUERY_IMAGE:
    raise RuntimeError('Both gallery and query images are required.')

In [ ]:
# Load and detect
g_img = cv2.imread(GALLERY_IMAGE); q_img = cv2.imread(QUERY_IMAGE)
g_faces = detector.process_frame(g_img)
q_faces = detector.process_frame(q_img)
print(f'Gallery faces: {len(g_faces)} | Query faces: {len(q_faces)}')

In [ ]:
# Collect embeddings and crops
g_embeddings = [f['embedding'].astype(np.float32) for f in g_faces if f.get('embedding') is not None]
g_crops = [cv2.cvtColor(f['face_img'], cv2.COLOR_BGR2RGB) for f in g_faces]
q_embeddings = [f['embedding'].astype(np.float32) for f in q_faces if f.get('embedding') is not None]
q_crops = [cv2.cvtColor(f['face_img'], cv2.COLOR_BGR2RGB) for f in q_faces]

if not g_embeddings or not q_embeddings:
    raise RuntimeError('Embeddings missing for gallery or query faces. Ensure InsightFace recognition models are loaded.')

In [ ]:
# Compute similarity matrix
sim = cosine_sim(np.stack(q_embeddings), np.stack(g_embeddings))  # shape (Q, G)
print('Similarity matrix shape:', sim.shape)

In [ ]:
# Visualize best matches for each query face
THRESH = 0.35  # similarity threshold for declaring a match (tune for your model)

for qi in range(sim.shape[0]):
    best_idx = int(np.argmax(sim[qi]))
    best_score = float(sim[qi, best_idx])
    matched = best_score >= THRESH
    fig, axs = plt.subplots(1,2, figsize=(8,4))
    axs[0].imshow(q_crops[qi]); axs[0].set_title(f'Query #{qi+1}'); axs[0].axis('off')
axs[1].imshow(g_crops[best_idx])
axs[1].set_title(f'Gallery #{best_idx+1}\nScore: {best_score:.3f} -> {"MATCH" if matched else "UNFAMILIAR"}')
axs[1].axis('off')
plt.show()

**Exercise:** Try varying `THRESH`. Lower thresholds increase false positives; higher thresholds increase false negatives. For production systems validate threshold on a labeled dataset.